In [1]:
!python /Users/kittnguyen/Documents/DS201_Finance/src/training/train_textcnn.py

Device being used: mps
Loading vocab ... 
Building vocab from /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/train_vifinner.jsonl
File /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/train_vifinner.jsonl có vẻ là JSON Lines. Đang chuyển sang chế độ đọc từng dòng...
Đã load xong dữ liệu
Vocab size: 25879
Num tags: 17
Loading dataset ... 
Loading data from /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/train_vifinner.jsonl
File /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/train_vifinner.jsonl có vẻ là JSON Lines. Đang chuyển sang chế độ đọc từng dòng...
Đã load xong dữ liệu
Loading data from /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/dev_vifinner.jsonl
File /Users/kittnguyen/Documents/DS201_Finance/data/labeled/ner/syllables/dev_vifinner.jsonl có vẻ là JSON Lines. Đang chuyển sang chế độ đọc từng dòng...
Đã load xong dữ liệu
Loading data from /Users/kittnguyen/Documents/DS20

In [1]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

nlp = pipeline("ner", model=model, tokenizer=tokenizer)
example = "My name is Wolfgang and I live in Berlin"

ner_results = nlp(example)
print(ner_results)

/Users/kittnguyen/Documents/DS201_Finance/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


[{'entity': 'B-PER', 'score': np.float32(0.9990139), 'index': 4, 'word': 'Wolfgang', 'start': 11, 'end': 19}, {'entity': 'B-LOC', 'score': np.float32(0.999645), 'index': 9, 'word': 'Berlin', 'start': 34, 'end': 40}]


In [3]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load tokenizer và model XLM-RoBERTa
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
model = AutoModel.from_pretrained("xlm-roberta-base")

# Ví dụ tokens đã được tách (word-level tokens)
tokens = ["Đây", "là", "một", "ví", "dụ", "cho", "NER", "."]

# Nối tokens thành câu để tokenizer xử lý
sentence = " ".join(tokens)

# Tokenize và lấy word_ids để map subtoken về token gốc
inputs = tokenizer(sentence, return_tensors="pt")

# Lấy danh sách word_ids (token gốc ứng với mỗi subtoken)
word_ids = inputs.word_ids(batch_index=0)

with torch.no_grad():
    outputs = model(**inputs)
    last_hidden_states = outputs.last_hidden_state  # (1, seq_len, hidden_dim)

# Map embedding từng subtoken về token gốc trung bình
token_embeddings = []
current_word = None
current_embeddings = []

for idx, word_id in enumerate(word_ids):
    if word_id is None:
        continue
    if word_id != current_word:
        if current_embeddings:
            # Trung bình embedding các subtoken thuộc token gốc trước đó
            token_emb = torch.stack(current_embeddings).mean(dim=0)
            token_embeddings.append(token_emb)
            current_embeddings = []
        current_word = word_id
    current_embeddings.append(last_hidden_states[0, idx])

# Thêm embedding token cuối cùng
if current_embeddings:
    token_emb = torch.stack(current_embeddings).mean(dim=0)
    token_embeddings.append(token_emb)

print(f"Số tokens gốc: {len(tokens)}")
print(f"Số embeddings lấy được: {len(token_embeddings)}")

# In shape embedding từng token (hidden size thường là 768)
for i, emb in enumerate(token_embeddings):
    print(f"Token: {tokens[i]}, Embedding shape: {emb.shape}")

Số tokens gốc: 8
Số embeddings lấy được: 8
Token: Đây, Embedding shape: torch.Size([768])
Token: là, Embedding shape: torch.Size([768])
Token: một, Embedding shape: torch.Size([768])
Token: ví, Embedding shape: torch.Size([768])
Token: dụ, Embedding shape: torch.Size([768])
Token: cho, Embedding shape: torch.Size([768])
Token: NER, Embedding shape: torch.Size([768])
Token: ., Embedding shape: torch.Size([768])
